In [17]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import zipfile

## Function Documentation: `compute_metrics`

This function serves as the central evaluation engine for the experimental results. It incorporates the true targets and the model's outputs to compute a wide range of standard classification, ordinal classification, and class-specific metrics.

### 📋 Parameters
* **`targets`** *(array-like)*: The ground-truth class labels for the evaluation set.
* **`probabilities`** *(array-like)*: The output from the model. This can either be raw logits/probabilities (a 2D matrix of shape `[n_samples, n_classes]`) or hard class predictions (a 1D array of shape `[n_samples]`).

---

### ⚙️ How It Works (Step-by-Step)

#### 1. Data Standardisation & Hard Label Extraction
The function converts inputs into NumPy arrays. It then automatically checks the dimensions of the `probabilities` argument:
* **If 2D (Probabilities/Logits):** It extracts the hard predictions using an `argmax` operation along the class axis (`axis=1`).
* **If 1D (Hard Labels):** It treats the inputs directly as the final predicted classes.

#### 2. Global Metrics Calculation
It evaluates model performance across several standard and ordinal-specific metrics:
* **`QWK`**: Quadratic Weighted Kappa (rewards predictions that are close to the true ordinal rank).
* **`MAE`**: Mean Absolute Error (measures the average absolute distance between predicted and true ordinal classes).
* **`1-off`**: 1-Off Accuracy (the accuracy allowing a maximum error margin of $\pm1$ class deviation).
* **`CCR`**: Correct Classification Rate (Standard Accuracy).
* **`MS`**: Minimum Sensitivity across all classes.
* **`BalancedAccuracy`**: Balanced Accuracy score (unweighted mean of per-class recall).
* **`AMAE` / `MMAE`**: Macro-averaged metrics specific to Ordinal Classification.
* **`RPS`**: Rank Probability Score (evaluates the quality of the predicted probability distribution across ordered categories, applying a `softmax` to logits first).
* **`MES` / `GMES`**: Specialized classification metrics.

#### 3. Class-Specific Sensitivity (Recall) Extraction
The function computes the individual recall score for every class using `recall_score(..., average=None)`. It dynamically injects these into the output dictionary under the naming convention **`SensX`** (where `X` represents the class index).

---

### 📥 Returns
* **`metrics`** *(dict)*: A dictionary containing all global metrics alongside the individual class sensitivities (`Sens0`, `Sens1`, etc.).

In [18]:
def compute_metrics(targets, probabilities):
    from dlordinal.metrics import accuracy_off1, minimum_sensitivity
    from scipy.special import softmax
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        cohen_kappa_score,
        mean_absolute_error,
        recall_score,
    )

    from experiments.metrics import amae, gmes, mes, mmae, rank_probability_score

    targets = np.array(targets)
    probabilities = np.array(probabilities)

    if len(probabilities.shape) > 1:
        predictions = np.argmax(probabilities, axis=1)
    else:
        predictions = np.array(probabilities)

    metrics = {
        "QWK": cohen_kappa_score(targets, predictions, weights="quadratic"),
        "MAE": mean_absolute_error(targets, predictions),
        "1-off": accuracy_off1(targets, predictions),
        "CCR": accuracy_score(targets, predictions),
        "MS": minimum_sensitivity(targets, predictions),
        "BalancedAccuracy": balanced_accuracy_score(targets, predictions),
        "AMAE": amae(targets, predictions),
        "MMAE": mmae(targets, predictions),
        "RPS": rank_probability_score(targets, softmax(probabilities, axis=1)),
        "MES": mes(targets, predictions),
        "GMES": gmes(targets, predictions),
    }

    # Compute sensitivities for each class
    sensitivities = np.array(recall_score(targets, predictions, average=None))

    for i, sens in enumerate(sensitivities):
        metrics[f"Sens{i}"] = sens

    return metrics

## Function Documentation: `_process_single_seed_worker`

This standalone helper function acts as the **isolated processing unit** for the pipeline's parallel execution engine. It handles the extraction, transformation, and evaluation of a single experimental seed configuration independently on a dedicated CPU core.

### 🧠 Parallel Architecture Note: Why is it outside the class?
Python's `ProcessPoolExecutor` relies on **multiprocessing**, meaning it distributes computational tasks across separate CPU cores. To do this, it must serialize and transmit data structures across processes using a mechanism called **Pickling**. 
* Regular instance methods (`self.method`) cannot be easily pickled because they are tied to the complete internal state of the class instance.
* Leaving this function as a **top-level, global helper** allows Python to smoothly pickle and distribute it across all available CPU worker cores without overhead or memory locking issues.

---

### 📋 Input Parameters
* **`args_tuple`** *(tuple)*: A bundled collection containing five distinct parameters required for processing:
  1. `test_file` *(Path)*: Destination path pointing to the specific `test_predictions.csv` target.
  2. `estimator_name` *(str)*: Raw machine learning model identification string.
  3. `dataset_name` *(str)*: Name of the dataset under evaluation.
  4. `rs` *(int)*: The Random Seed integer index (e.g., `0` through `29`).
  5. `model_mapping` *(dict)*: Dictionary mapping used to substitute raw model labels with publication-ready names.

---

### ⚙️ Operational Workflow

#### 1. File Ingestion & Validation
The worker attempts to load the targeted file into a temporary Pandas DataFrame. If the file is entirely empty or missing features, it gracefully aborts early returning `None`, preventing computing resources from being wasted.

#### 2. Label & Probability Transformation
The worker maps the critical target values (`y_true`) and prepares the predictions:
* **The String Array Challenge:** In plain CSV formats, lists of model prediction probabilities are written down as basic text strings like `"[0.15, 0.45, 0.40]"`. 
* **The Solution:** The function dynamically strips away the bracket elements and uses `np.fromstring(..., sep=',')` to reconstitute that flat text into raw, multi-column NumPy array vectors. If probabilities are absent, it safely falls back to standard hard predicted labels (`Prediction`).

#### 3. Metrics Generation & Metadata Injection
The standard targets and normalized predictions are passed to `compute_metrics()`. Once the dictionary of results is compiled, the worker appends structural logging identifiers (`dataset`, `estimator_name`, and `rs`) directly into the record.

#### 4. Fault Tolerance
The entire script execution is wrapped inside a robust `try-except` layer. If a single seed CSV file is corrupt, truncated, or broken, the worker traps the error message and forwards it as a structured warning instead of crashing the entire background processing grid.

---

### 📥 Returns
* **`dict_metrics`** *(dict)*: An active dictionary holding all raw evaluations alongside structural execution tags.
* **`None`**: Returned when encountering an empty or broken datafile.
* **`{"error": ...}`** *(dict)*: Error diagnostic log packet returned upon failure.

In [19]:
def _process_single_seed_worker(args_tuple):
    """
    Worker function executed in parallel across CPU cores.
    Processes a single prediction CSV file and computes metrics.
    """
    test_file, estimator_name, dataset_name, rs, model_mapping = args_tuple
    try:
        df_pred = pd.read_csv(test_file)
        if df_pred.empty:
            return None
            
        y_true = df_pred['Target'].values
        
        # Parse prediction probabilities string back into a numeric NumPy array
        if 'Prediction probabilities' in df_pred.columns:
            y_pred = df_pred['Prediction probabilities'].apply(
                lambda x: np.fromstring(x.strip('[]'), sep=',')
            ).values
            y_pred = np.stack(y_pred)
        else:
            y_pred = df_pred['Prediction'].values
            
        # Compute metrics using the external experimental utility
        dict_metrics = compute_metrics(y_true, y_pred)
        
        # Inject structural metadata attributes
        dict_metrics['dataset'] = dataset_name
        dict_metrics['estimator_name'] = model_mapping.get(estimator_name.lower(), estimator_name)
        dict_metrics['rs'] = rs
        return dict_metrics
        
    except Exception as e:
        return {"error": f"Error in {dataset_name} ({estimator_name}) seed_{rs}: {e}"}

In [20]:

class MetricsReporter:
    """
    A class to automatically discover machine learning prediction files,
    compute custom metrics in parallel, aggregate seed results, and generate
    formatted Excel workbooks along with compressed directory backups.
    """
    def __init__(self, base_path="./results_tocuco", output_excel="final_metrics_report.xlsx", model_mapping=None):
        self.base_path = Path(base_path)
        self.output_excel = output_excel
        self.performance_records = []
        
        # Default fallback model mapping if none is provided
        self.model_mapping = model_mapping if model_mapping is not None else {
            "logatcwclassifier": "LogAT-cwBal",
            "logitcwclassifier": "LogIT-cwBal",
            "mlpclmclassifier": "MLP-CLM",
            "mlpclassifier": "MLP",
            "mlptriangularclassifier": "MLP-T",
        }

    def _discover_tasks(self):
        """Scans the directory structure to collect valid file path payloads."""
        tasks = []
        if not self.base_path.exists():
            print(f"❌ ERROR: Target directory '{self.base_path}' does not exist.")
            return tasks

        for model_dir in self.base_path.iterdir():
            if model_dir.is_dir():
                estimator_name = model_dir.name
                for dataset_dir in model_dir.iterdir():
                    if dataset_dir.is_dir():
                        dataset_name = dataset_dir.name
                        pred_by_seed_dir = dataset_dir / "predictions_by_seed"
                        
                        if not pred_by_seed_dir.exists():
                            continue
                        
                        for seed_dir in pred_by_seed_dir.iterdir():
                            if seed_dir.is_dir() and seed_dir.name.startswith("seed_"):
                                try:
                                    rs = int(seed_dir.name.split("_")[1])
                                except ValueError:
                                    continue
                                
                                test_file = seed_dir / "test_predictions.csv"
                                if test_file.exists():
                                    # Passing parameters along with mapping dictionary for the worker
                                    tasks.append((test_file, estimator_name, dataset_name, rs, self.model_mapping))
        return tasks

    def run_pipeline(self):
        """Executes the complete end-to-end ingestion, computation, and export pipeline."""
        print(f"Scanning results directory: {self.base_path.resolve()}")
        tasks = self._discover_tasks()
        
        if not tasks:
            print("❌ No valid target prediction files found. Process aborted.")
            return
            
        print(f"✨ Found {len(tasks)} target prediction files ready for ingestion.")
        print("🚀 Running data processing in parallel using all available CPU workers...")
        
        # Multiprocessing execution pool
        with ProcessPoolExecutor(max_workers=None) as executor:
            execution_results = list(executor.map(_process_single_seed_worker, tasks))
        
        # Filter and log errors from parallel workers
        self.performance_records = []
        for record in execution_results:
            if record is not None:
                if "error" in record:
                    print(f"⚠️ {record['error']}")
                else:
                    self.performance_records.append(record)
                    
        print(f"🔍 Successfully ingested and structured records: {len(self.performance_records)}")
        
        # Generate the spreadsheet dataframes if records exist
        if self.performance_records:
            self._generate_excel()
        else:
            print("❌ No successful metric records collected. Excel generation skipped.")

    def _generate_excel(self):
        """Processes flat metric data into aggregated dataframes and writes them to Excel."""
        df_individual = pd.DataFrame(self.performance_records)
        
        # Schema sorting
        ordered_columns = ['dataset', 'estimator_name', 'rs'] + [col for col in df_individual.columns if col not in ['dataset', 'estimator_name', 'rs']]
        df_individual = df_individual[ordered_columns]
        df_individual.sort_values(by=["dataset", "estimator_name", "rs"], inplace=True)
        
        # Compute mean performances across seeds
        group_cols = ['dataset', 'estimator_name']
        df_average = df_individual.groupby(group_cols).mean().reset_index()
        if 'rs' in df_average.columns:
            df_average.drop(columns=['rs'], inplace=True)
            
        # Filter out sensitivity ('Sens') metrics dynamically
        detected_metrics = [col for col in df_average.columns if col not in group_cols and not col.lower().startswith('sens')]
        clean_individual_cols = [col for col in df_individual.columns if not col.lower().startswith('sens')]
        
        df_individual = df_individual[clean_individual_cols]
        df_average = df_average[group_cols + detected_metrics]

        # Secure spreadsheet compilation
        with pd.ExcelWriter(self.output_excel, engine='openpyxl') as writer:
            df_individual.round(3).to_excel(writer, sheet_name="Individual", index=False)
            df_average.round(3).to_excel(writer, sheet_name="Average", index=False)
            
            # Append cross-tabulated metric sheets
            for metric in detected_metrics:
                df_pivoted = df_average.pivot(index='dataset', columns='estimator_name', values=metric).round(3)
                df_pivoted.index.name = 'Dataset'
                df_pivoted.columns.name = None
                df_pivoted.to_excel(writer, sheet_name=metric)
                
        print(f"✅ Metric workbook generated and saved as: '{self.output_excel}'")

    def compress_results(self, output_zip="results_tocuco.zip"):
        """
        Compresses the source results directory into a ZIP archive.
        Optimized to skip temporary lock files or system hidden files.
        """
        if not self.base_path.exists():
            print(f"❌ Cannot compress: Target directory '{self.base_path}' does not exist.")
            return

        print(f"📦 Compressing source directory to '{output_zip}' (optimized mode)...")
        try:
            with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
                for file_path in self.base_path.rglob("*"):
                    # Ignore temporary locks or hidden operating system files
                    if "lock" in file_path.name or file_path.name.startswith("."):
                        continue
                    if file_path.is_file():
                        z.write(file_path, file_path.relative_to(self.base_path.parent))
                        
            print(f"✅ Archive compression successfully completed: '{output_zip}'")
        except Exception as e:
            print(f"⚠️ Archive compression failed: {e}")

## Implementation Guide: Running the `MetricsReporter` Pipeline

To achieve the exact same outputs as your original standalone script—generating the structured multi-sheet Excel file (`final_metrics_report.xlsx`)—you simply need to instantiate the class and invoke its execution method.

### 🏃‍♂️ Standard Execution (Matches Your Original Setup)

Place this code block into a fresh cell in your Jupyter Notebook and run it. It assumes your target folder is named `results_tocuco` and sits in the same directory as your notebook:

```python
# 1. Initialise the reporter with default settings
reporter = MetricsReporter(
    base_path="./results_tocuco", 
    output_excel="final_metrics_report.xlsx"
)

# 2. Run the ingestion, multi-core calculation, and Excel generation pipeline
reporter.run_pipeline()
```

In [21]:
# 1. Instatiate the class
reporter = MetricsReporter(
    base_path="./results_tocuco", 
    output_excel="final_metrics_report.xlsx"
)

# 2. Generate your sheets ("Individual", "Average", and pivot tabs)
reporter.run_pipeline()

# 3. Create the optimized ZIP file skipping lock files
reporter.compress_results("results.zip")

Scanning results directory: /home/fberchez/DeepLearning/tocuco/results_tocuco
✨ Found 11040 target prediction files ready for ingestion.
🚀 Running data processing in parallel using all available CPU workers...
🔍 Successfully ingested and structured records: 11040
✅ Metric workbook generated and saved as: 'final_metrics_report.xlsx'
📦 Compressing source directory to 'results.zip' (optimized mode)...
✅ Archive compression successfully completed: 'results.zip'
